# Pipefy — Transferência em massa de responsável

Reatribui o **responsável (assignee)** de **todos os cards abertos**, em **todos os pipes**, de um usuário para outro — via API GraphQL do Pipefy.

Feito para o cenário clássico: alguém sai do time e deixa dezenas de cards espalhados. Em vez de reatribuir na mão, um card de cada vez, o script varre a conta inteira e faz a troca de forma **reproduzível e auditável**.

**O que ele tem de cuidado:**
- **Modo de simulação (`DRY_RUN`)**: lista tudo o que *seria* alterado antes de tocar em qualquer card.
- **Preserva co-responsáveis**: a mutation `updateCard` substitui a lista inteira de responsáveis, então o script remove apenas o antigo e mantém os demais.
- **Respeita o rate limit**: paginação de 50 em 50, pausas entre chamadas e *backoff* automático.

> **Segurança:** o token é lido da variável de ambiente `PIPEFY_TOKEN` — nunca fica escrito no notebook. Veja o `README.md`.

**Ordem de uso:** conexão → descobrir IDs → listar pipes → configurar → prévia → executar.

In [ ]:
import os
import time
import json
import requests

# Opcional: carrega o token de um arquivo .env, se existir (pip install python-dotenv)
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

PIPEFY_TOKEN = os.environ.get("PIPEFY_TOKEN", "")
API_URL = "https://api.pipefy.com/graphql"
SLEEP = 0.3  # pausa entre chamadas, p/ respeitar o rate limit

if not PIPEFY_TOKEN:
    raise RuntimeError(
        "Token nao encontrado. Defina a variavel de ambiente PIPEFY_TOKEN "
        "(ex.: export PIPEFY_TOKEN='seu_token') ou crie um arquivo .env. Veja o README."
    )

HEADERS = {
    "Authorization": f"Bearer {PIPEFY_TOKEN}",
    "Content-Type": "application/json",
}
print("Conexao configurada.")

## Função base de chamada à API

Faz a requisição, trata erros GraphQL e aplica *backoff* exponencial quando bate no rate limit (HTTP 429 ou erro de limite no corpo da resposta).

In [ ]:
def run_query(query, variables=None, max_retries=5):
    for attempt in range(max_retries):
        resp = requests.post(
            API_URL, headers=HEADERS,
            json={"query": query, "variables": variables or {}},
        )

        if resp.status_code == 429:
            wait = 2 ** attempt
            print(f"  Rate limit (HTTP 429). Aguardando {wait}s...")
            time.sleep(wait)
            continue

        try:
            data = resp.json()
        except ValueError:
            raise RuntimeError(f"Resposta nao-JSON ({resp.status_code}): {resp.text[:300]}")

        if data.get("errors"):
            msgs = "; ".join(e.get("message", "") for e in data["errors"])
            if any(k in msgs.lower() for k in ("rate", "limit", "too many")):
                wait = 2 ** attempt
                print(f"  Limite via GraphQL: {msgs}. Aguardando {wait}s...")
                time.sleep(wait)
                continue
            raise RuntimeError("Erro GraphQL: " + json.dumps(data["errors"], ensure_ascii=False))

        return data["data"]

    raise RuntimeError("Numero maximo de tentativas excedido (rate limit).")

## 1) Descobrir IDs de usuários

Lista os membros de cada organização com `id`, nome e e-mail. Use para preencher `OLD_USER_ID` e `NEW_USER_ID` na configuração.

> Se este campo não retornar o esperado, você também consegue o ID abrindo o perfil da pessoa no Pipefy e olhando o número na URL.

In [ ]:
def listar_usuarios():
    query = """
    query {
      organizations {
        id
        name
        members { user { id name email } }
      }
    }
    """
    data = run_query(query)
    for org in data["organizations"]:
        print(f"\nOrganizacao: {org['name']} (id {org['id']})")
        usuarios = {}
        for m in org.get("members", []) or []:
            u = m.get("user")
            if u:
                usuarios[u["id"]] = u
        for uid, u in sorted(usuarios.items(), key=lambda x: (x[1].get("name") or "").lower()):
            print(f"  id={uid:>10}  {u.get('name','')}  <{u.get('email','')}>")

listar_usuarios()

## 2) Listar pipes

Mostra todos os pipes visíveis pelo token — exatamente o conjunto que será varrido (a menos que use `PIPE_IDS_FILTER`).

In [ ]:
def listar_pipes():
    query = """
    query {
      organizations {
        id
        name
        pipes { id name }
      }
    }
    """
    data = run_query(query)
    pipes = []
    for org in data["organizations"]:
        for p in org.get("pipes", []) or []:
            pipes.append({"id": p["id"], "name": p["name"], "org": org["name"]})
    return pipes

pipes = listar_pipes()
print(f"{len(pipes)} pipe(s) encontrado(s):")
for p in pipes:
    print(f"  id={p['id']:>10}  {p['name']}  ({p['org']})")

## 3) Configuração da transferência

Preencha os IDs (vistos no passo 1). Mantenha `DRY_RUN = True` até revisar a prévia.

In [ ]:
OLD_USER_ID = "OLD_USER_ID"   # responsavel atual — sera REMOVIDO
NEW_USER_ID = "NEW_USER_ID"   # novo responsavel — sera ADICIONADO

DRY_RUN = True           # True = so simula. Mude p/ False para executar de verdade.
PIPE_IDS_FILTER = []     # vazio = todos os pipes; ou liste IDs especificos.

print(f"De {OLD_USER_ID} -> {NEW_USER_ID} | DRY_RUN = {DRY_RUN}")

## 4) Funções de busca e transferência

- `cards_do_responsavel`: pagina o pipe (50 por página) e retorna só os cards **abertos** (`done = false`) que têm o responsável antigo.
- `transferir_responsavel`: monta a nova lista (remove o antigo, mantém os demais, adiciona o novo) e chama `updateCard`.

In [ ]:
def cards_do_responsavel(pipe_id, old_user_id):
    """Cards ABERTOS (done=false) do pipe que tem old_user_id como responsavel."""
    query = """
    query($pipeId: ID!, $after: String) {
      allCards(pipeId: $pipeId, first: 50, after: $after) {
        pageInfo { hasNextPage endCursor }
        edges {
          node {
            id
            title
            done
            current_phase { name }
            assignees { id name }
          }
        }
      }
    }
    """
    resultado, cursor = [], None
    while True:
        data = run_query(query, {"pipeId": pipe_id, "after": cursor})
        conn = data["allCards"]
        for edge in conn["edges"]:
            node = edge["node"]
            if node["done"]:
                continue  # ignora cards concluidos
            ids = [a["id"] for a in node["assignees"]]
            if old_user_id in ids:
                resultado.append(node)
        if conn["pageInfo"]["hasNextPage"]:
            cursor = conn["pageInfo"]["endCursor"]
            time.sleep(SLEEP)
        else:
            break
    return resultado


def transferir_responsavel(card, old_user_id, new_user_id):
    ids = [a["id"] for a in card["assignees"]]
    novos = [i for i in ids if i != old_user_id]   # remove o antigo, mantem os outros
    if new_user_id not in novos:
        novos.append(new_user_id)                  # adiciona o novo (sem duplicar)

    mutation = """
    mutation($id: ID!, $assignees: [ID]) {
      updateCard(input: { id: $id, assignee_ids: $assignees }) {
        card { id assignees { id name } }
      }
    }
    """
    data = run_query(mutation, {"id": card["id"], "assignees": novos})
    return data["updateCard"]["card"]

## 5) Prévia — quantos cards e em quais pipes (não altera nada)

Escaneia todos os pipes-alvo e mostra uma tabela ordenada (mais cards primeiro) + o total. Guarda o resultado em `PREVIEW`, que a execução reaproveita para não buscar duas vezes.

In [ ]:
pipes = listar_pipes()  # garante lista atualizada
alvos = pipes if not PIPE_IDS_FILTER else [p for p in pipes if p["id"] in PIPE_IDS_FILTER]

PREVIEW = {}          # cache p/ reaproveitar na execucao (evita buscar 2x)
resumo = []
total = 0

print(f"Escaneando {len(alvos)} pipe(s) em busca de cards de {OLD_USER_ID}...\n")
for p in alvos:
    try:
        cards = cards_do_responsavel(p["id"], OLD_USER_ID)
    except Exception as e:
        print(f"  [erro] {p['name']} (id {p['id']}): {e}")
        continue
    if cards:
        PREVIEW[p["id"]] = cards
        resumo.append((p["name"], p["id"], len(cards)))
        total += len(cards)

resumo.sort(key=lambda x: x[2], reverse=True)  # mais cards primeiro

print(f"{'Pipe':<40} {'ID':>12} {'Cards':>7}")
print("-" * 61)
for nome, pid, n in resumo:
    print(f"{nome[:40]:<40} {pid:>12} {n:>7}")
print("-" * 61)
print(f"{'TOTAL':<40} {'':>12} {total:>7}")
print(f"\n{total} card(s) aberto(s) em {len(resumo)} pipe(s) seriam transferidos.")

## 6) Executar

Reaproveita a varredura da prévia (`PREVIEW`). Com `DRY_RUN = True`, apenas imprime; com `False`, aplica de fato.

In [ ]:
alvos = pipes if not PIPE_IDS_FILTER else [p for p in pipes if p["id"] in PIPE_IDS_FILTER]

total_encontrados = 0
total_transferidos = 0
falhas = []

print(f"{'SIMULACAO (nada sera alterado)' if DRY_RUN else 'EXECUCAO REAL'} — {len(alvos)} pipe(s)\n")

for p in alvos:
    print(f"Pipe: {p['name']} (id {p['id']})")
    try:
        # usa o cache da previa; se nao houver, busca na hora
        cards = PREVIEW.get(p["id"])
        if cards is None:
            cards = cards_do_responsavel(p["id"], OLD_USER_ID)
    except Exception as e:
        print(f"   [erro] busca de cards: {e}")
        falhas.append(("busca", p["id"], str(e)))
        continue

    if not cards:
        print("   nenhum card aberto com esse responsavel.")
        continue

    print(f"   {len(cards)} card(s) aberto(s) com o responsavel atual:")
    for c in cards:
        total_encontrados += 1
        fase = c["current_phase"]["name"] if c["current_phase"] else "?"
        print(f"     - [{c['id']}] {c['title']}  (fase: {fase})")
        if DRY_RUN:
            continue
        try:
            transferir_responsavel(c, OLD_USER_ID, NEW_USER_ID)
            total_transferidos += 1
            time.sleep(SLEEP)
        except Exception as e:
            print(f"        [erro] transferencia: {e}")
            falhas.append(("update", c["id"], str(e)))

print("\n" + "=" * 55)
if DRY_RUN:
    print(f"SIMULACAO: {total_encontrados} card(s) seriam alterados.")
    print("Se estiver correto, mude DRY_RUN=False na configuracao e rode de novo.")
else:
    print(f"CONCLUIDO: {total_transferidos}/{total_encontrados} card(s) transferido(s).")
if falhas:
    print(f"\n{len(falhas)} falha(s):")
    for tipo, ident, msg in falhas:
        print(f"  ({tipo}) {ident}: {msg}")